# BP6 Gate 1 — Business Understanding & Policy
**Customer360 Navigator Enterprise Suite — GenAI Resolution Assistant**

## Why this notebook is a genuinely different animal from BP1–BP5's own Gate 1 notebooks
Every prior BP's Gate 1 notebook (BP1–BP4, real-run confirmed; BP5, currently being built) defines a
**supervised target or a descriptive unit of analysis** over data BP1–BP5 already has in hand. BP6 has
neither. Per Master Plan Section 5.1: *"BP6 — GenAI Resolution Assistant: retrieve structured evidence
(from BP1–BP5 outputs), summarize and recommend a next action with citations/evidence fields on every
claim; human-in-the-loop approval gate before any recommendation is treated as final."* BP6's actual
GenAI call is Gate 5's job, not Gate 1's (Section 8's gate table: Gate 5 = "Decision / GenAI Layer &
Reporting", output = "Grounded GenAI output... every claim carries a citation/evidence field"). **This
notebook makes zero external API calls, loads zero GenAI SDK, and retrieves zero evidence.** Its job is
narrower and comes first: define, in writing and verified live against the real project state, (a)
which upstream BPs BP6 will eventually retrieve evidence from and which of those real artifacts exist
today, and (b) the governance policy every future GenAI call in this suite must satisfy before Gate 5
may treat any recommendation as final — citation/evidence schema, human-in-the-loop, UDAAP language
review, NIST AI RMF risk category, and GLBA PII masking. This is why this notebook's own policy.json
looks structurally different from BP1's (`target_definition` / `leakage_rules`) and BP4's
(`journey_definition` / `scope_boundaries`): BP6 is a retrieval-and-generation governance layer, not a
classification or aggregation problem, and this notebook does not force either of those schemas onto it
(see Section 7 below for the schema actually used, and the reasoning for it).

## Why the upstream-dependency check does *not* require BP1–BP5's outputs to exist yet
Gate 1's job, per Section 8, is "Business Understanding & Policy" with exit criterion "No target leakage
possible by construction" — for BP6 this is honestly reframed (Section 7 below) as *no ungrounded or
unapproved recommendation can be treated as final, by construction*. Defining **scope and dependency
policy** is exactly Gate 1's job; it does not require the dependency to be delivered yet, any more than
BP1's Gate 1 required BP2's taxonomy work to exist before BP1 could state it would eventually feed BP8.
This notebook therefore states BP6's real upstream dependency on BP1–BP5 as **policy**, then verifies
*live, against the real project filesystem* — not from memory, not assumed — exactly which of those five
BPs have a real, already-delivered Gate 1 `policy.json` artifact today and which do not. As of this
notebook's own most recent edit, BP1–BP4 each have one; BP5 does not yet (a separate, concurrent Gate 1
build). **This is reported honestly below as a live filesystem check, not hardcoded** — if BP5's Gate 1
lands before this notebook is next run, the live check will show it, without anyone touching this code.

## The schema-agnostic evidence design (why BP6 never hardcodes BP1–BP5's internal schemas)
Even where an upstream BP's real artifact already exists today (BP1–BP4), this notebook deliberately
does **not** hardcode any of their internal field names into BP6's own evidence schema. BP1's Gate 1
`policy.json` records `target_definition.primary_target = "category"`; BP4's records
`journey_definition.unit_1_complaint_event_journey`. Baking either shape into BP6's citation schema now
would (a) misrepresent Gate 1's own job — BP6 doesn't consume those fields until its own Gate 5 retrieval
step exists — and (b) silently break the moment an upstream BP's schema evolves between now and BP6
Gate 5 (a real risk: BP2's Gate 1 already narrowed its own taxonomy after being written, and BP5's Gate 1
is being built concurrently with this one). Instead, BP6's citation/evidence schema (Section 7) is a
generic wrapper — `source_bp`, `source_gate`, `source_artifact_relative_path`, `source_field_or_metric`,
`extracted_value`, `retrieval_timestamp_utc`, `verification_method` — that can point at *any* upstream
artifact and field, present or future, without BP6 ever needing to know that artifact's shape in advance.
Every upstream BP's own evidence schema is recorded below as `TBD_PENDING_<BP>_OWN_GATE6_DELIVERY`
(or the literal status string this notebook actually reads live from that BP's own config file) — never
guessed.

## Business Understanding (Master Plan Section 5.1 / 7, BP6)
BP6 retrieves structured evidence from BP1–BP5's real outputs (once those exist), summarizes it, and
recommends a next action — every claim carrying a citation/evidence field, no recommendation ever
auto-applied without a human approving it first. Per Master Plan Section 9's compliance table, three
regulatory frameworks name BP6 specifically: **UDAAP** ("every GenAI-drafted customer-facing
recommendation is reviewed for deceptive/misleading language before human approval, Gate 5"), **NIST AI
RMF 1.0** ("BP6 (GenAI) primarily... every GenAI output requires human-in-loop approval plus a documented
risk category"), and **GLBA** ("any PII surfaced in complaint narratives is screened and masked before
any GenAI call (BP6)"). This notebook converts each of those three sentences into a concrete, checkable
policy field (Section 7) rather than leaving them as prose to be remembered later. Per
`docs/data_dictionary/RAW_DATA_MANIFEST.md` Finding 2 (re-verified live below, Section 4, consistent with
every prior BP1–BP4 Gate 1 finding on the same real CFPB schema): **this project's real CFPB extract
carries no narrative/complaint-text column at all.** BP1 and BP6 "as currently scoped for CFPB text"
(the Manifest's own words) must instead rely on BANKING77's real `text` field for any narrative-text
input — meaning the GLBA PII-masking requirement below applies, in practice, to BANKING77 `text` values
surfaced through BP1's evidence, not to any CFPB narrative (there is none to mask).

## Standing rules this notebook follows
- **Execution boundary** (Section 12.2): Claude wrote this notebook; it does not run it. You run it on
  your own machine, and the real, live-checked results below become this project's Gate 1 policy record
  for BP6.
- **Zero-fabrication** (Section 12.1): every check below runs against the real files in `data/external/`,
  the real `configs/*.yaml` status fields of BP1–BP5, and the real presence/absence of each upstream BP's
  own `notebooks/<bp>/artifacts/policy.json`. No BP1–BP5 status, no BANKING77 count, no class-balance
  figure is asserted from memory.
- **No external call, ever, in this notebook**: no GenAI SDK is imported (verified live in Section 6 by
  checking `sys.modules` after every import this notebook performs); no HTTP client library is imported
  either. This is true by construction of this notebook's own source, and re-affirmed empirically at
  every run.
- **WARP**: `configure_performance()` first. BANKING77 is small (~13K rows) so this notebook loads it
  eagerly, exactly as BP1 Gate 1 does — no CFPB row-level data is loaded here at all (this notebook needs
  CFPB's *schema* only, for the narrative-text-absence and shared-column checks; the real 1,048,575-row
  file is never scanned by this notebook).
- **HYPER**: reuses `src/taxonomy/taxonomy_mapper.CFPB_DTYPES` (schema only) and
  `src/utils/bp1_config_sync.py` (front-matter read/write), exactly as BP1/BP4 do — nothing re-derived.
- **Idempotent**: re-running this notebook overwrites `configs/bp6_genai_resolution_assistant.yaml`
  (front matter only) and this notebook's own `policy.json` artifact in place.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same resolver as every other notebook in this project.

## Outputs (both written, idempotent overwrite-in-place)
- `configs/bp6_genai_resolution_assistant.yaml` — front matter only (`status`, `genai_governance_policy`,
  `upstream_dependency_status`, `assumptions`); any later gate's own marker-delimited block, once one
  exists, is preserved verbatim regardless of position, exactly as BP1's `write_front_matter` guarantees.
- `notebooks/bp6_genai_resolution_assistant/artifacts/policy.json` — the Section 8 Gate 1 output
  artifact: BP6's scope definition, upstream-dependency policy, GenAI governance policy (citation schema,
  human-in-the-loop, UDAAP, NIST AI RMF, GLBA), grounding-integrity rules, and live-checked evidence.

## Prerequisites
None of BP1–BP5's own gates need to have run for this notebook to complete successfully — that is the
whole point of Section 5's design (this notebook records what's available *today*, live, and is safe to
re-run at any later point as more upstream BPs land). `01_data_acquisition_profiling.ipynb` should have
been real-run at least once, since this notebook reads the real BANKING77 files it produced.

## If a structural check below fails
It raises `AssertionError` with the failing check named. The two checks inherited unchanged from BP1
(zero shared CFPB/BANKING77 columns, zero BANKING77 train/test text overlap) must never be worked
around — a nonzero value there is a real data-integrity problem, not a BP6-specific policy question. Any
policy-shape check (e.g. `human_in_the_loop_required_is_true`) failing means this notebook's own policy
dict was edited to violate the Master Plan's own compliance requirements for BP6 — fix the policy, never
the check.


In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp6_genai_resolution_assistant/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
ARTIFACTS_DIR = NOTEBOOKS_DIR / "bp6_genai_resolution_assistant" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import re  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import polars as pl  # noqa: E402

from taxonomy.taxonomy_mapper import CFPB_DTYPES  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

B77_TRAIN_PATH = DATA_EXTERNAL_DIR / "banking77_train.csv"
B77_TEST_PATH = DATA_EXTERNAL_DIR / "banking77_test.csv"
B77_CATEGORIES_PATH = DATA_EXTERNAL_DIR / "banking77_categories.json"

# BP6's own Gate 5 (its actual GenAI/decision gate, per Master Plan Section 8's gate table) is the
# only gate in this notebook's own BP that ever calls a GenAI API. This is a hardcoded, structural
# statement about BP6's design, checked empirically in Section 6 below (zero GenAI SDK modules
# loaded anywhere in this run).
GENAI_CALL_FIRST_OCCURS_AT_GATE = 5
FORBIDDEN_GENAI_SDK_MODULE_PREFIXES = ("openai", "anthropic", "google.generativeai", "cohere")

# ============================================================
# SECTION 4: Structural check - no shared identifier/join column between CFPB and BANKING77
# (the same real check BP1 Gate 1 runs; relevant to BP6 because its eventual retrieval step must
# reference CFPB-derived and BANKING77-derived evidence by each source's OWN real key, never a
# fabricated row-level join between the two schemas), plus a live re-verification that this real
# CFPB extract still carries no narrative-text column (RAW_DATA_MANIFEST.md Finding 2, re-verified
# by every BP1-BP4 Gate 1 to date). Both checks use CFPB's real column NAMES only - no CFPB
# row-level data is loaded by this notebook.
# ============================================================
cfpb_columns = set(CFPB_DTYPES.keys())
banking77_columns = set(pl.read_csv(B77_TRAIN_PATH, n_rows=1).columns)
shared_columns = cfpb_columns & banking77_columns
narrative_text_columns = sorted(c for c in cfpb_columns if "narrative" in c.lower())
print(f"[OK] CFPB columns ({len(cfpb_columns)}): {sorted(cfpb_columns)}")
print(f"[OK] BANKING77 columns: {sorted(banking77_columns)}")
print(f"[OK] Shared column names between the two schemas: {sorted(shared_columns) or 'NONE'}")
print(f"[OK] CFPB narrative-text columns: {narrative_text_columns or 'NONE'}")

# ============================================================
# SECTION 5: Real BANKING77 checks - the same dataset BP1's own real classifier is trained/
# evaluated on, and the only real narrative-text source BP6 will ever surface in a GenAI prompt
# (Section 4 above: CFPB carries none). Re-verified live here, not assumed from BP1's own policy.json.
# ============================================================
train = pl.read_csv(B77_TRAIN_PATH, schema_overrides={"text": pl.Utf8, "category": pl.Categorical})
test = pl.read_csv(B77_TEST_PATH, schema_overrides={"text": pl.Utf8, "category": pl.Categorical})
with open(B77_CATEGORIES_PATH, "r", encoding="utf-8") as f:
    b77_categories = json.load(f)

train_texts = set(train["text"].to_list())
test_texts = set(test["text"].to_list())
overlap_texts = train_texts & test_texts
print(f"[OK] BANKING77 train rows: {train.height:,}, test rows: {test.height:,}")
print(f"[OK] BANKING77 real category count: {len(b77_categories)}")
print(f"[OK] Exact-text overlap between train and test: {len(overlap_texts)} row(s)")

train_class_counts = train.group_by("category").agg(pl.len().alias("n")).sort("n", descending=True)
train_counts_list = train_class_counts["n"].to_list()
banking77_class_balance = {
    "n_classes": len(b77_categories),
    "n_classes_in_train": train_class_counts.height,
    "min_class_count": int(min(train_counts_list)) if train_counts_list else None,
    "max_class_count": int(max(train_counts_list)) if train_counts_list else None,
}
print(f"[OK] BANKING77 class balance (train split): {banking77_class_balance}")

# ============================================================
# SECTION 6: Zero-GenAI-SDK-loaded check - empirical confirmation that this notebook's own run
# never imported a GenAI client library, matching the "no external call, ever, in this notebook"
# standing rule stated in the markdown cell above.
# ============================================================
genai_sdk_modules_loaded = sorted(
    name for name in sys.modules if name.startswith(FORBIDDEN_GENAI_SDK_MODULE_PREFIXES)
)
print(f"[OK] GenAI SDK modules loaded in this run: {genai_sdk_modules_loaded or 'NONE'}")

# ============================================================
# SECTION 7: Live upstream-dependency check - BP1-BP5's real config status string and real Gate 1
# policy.json existence, read directly off the real filesystem, never hardcoded. This is what
# Section 5's markdown cell means by "reported honestly below as a live filesystem check."
# ============================================================
UPSTREAM_BPS = [
    ("bp1", "bp1_customer_intent_classification"),
    ("bp2", "bp2_customer_friction_classification"),
    ("bp3", "bp3_complaint_escalation_prediction"),
    ("bp4", "bp4_customer_journey_analytics"),
    ("bp5", "bp5_root_cause_driver_analytics"),
]
_STATUS_RE = re.compile(r'^status:\s*"([^"]*)"', flags=re.MULTILINE)
_GATE_CONFIRMED_RE = re.compile(r"gate(\d)_confirmed")


def _read_upstream_status(bp_id: str, bp_name: str) -> dict:
    config_path = CONFIGS_DIR / f"{bp_name}.yaml"
    policy_path = NOTEBOOKS_DIR / bp_name / "artifacts" / "policy.json"
    status_string = None
    if config_path.exists():
        match = _STATUS_RE.search(config_path.read_text(encoding="utf-8"))
        status_string = match.group(1) if match else None
    confirmed_gates = sorted(int(g) for g in _GATE_CONFIRMED_RE.findall(status_string or ""))
    if status_string and "gate6_complete" in status_string and 6 not in confirmed_gates:
        confirmed_gates.append(6)
    return {
        "bp_id": bp_id,
        "bp_name": bp_name,
        "config_status_string_live": status_string,
        "highest_confirmed_gate_live": max(confirmed_gates) if confirmed_gates else 0,
        "gate1_policy_artifact_exists_live": policy_path.exists(),
        "evidence_retrieval_readiness": (
            "ARTIFACT_AVAILABLE_TODAY" if policy_path.exists() else "PENDING_UPSTREAM_BP_DELIVERY"
        ),
        "evidence_schema_status": (
            f"OWN_REAL_SCHEMA_EXISTS_IN_{bp_name}_GATE1_POLICY_JSON_NOT_HARDCODED_HERE"
            if policy_path.exists()
            else f"TBD_PENDING_{bp_id.upper()}_OWN_GATE1_DELIVERY"
        ),
    }


upstream_dependency_policy = {bp_id: _read_upstream_status(bp_id, bp_name) for bp_id, bp_name in UPSTREAM_BPS}
for _bp_id, _row in upstream_dependency_policy.items():
    print(f"[OK] Upstream dependency {_bp_id}: {_row}")

# ============================================================
# SECTION 8: Assemble the Gate 1 policy - scope definition, upstream-dependency policy, GenAI
# governance policy (citation schema, human-in-the-loop, UDAAP, NIST AI RMF, GLBA), and
# grounding-integrity rules. This schema is deliberately NOT BP1's (target_definition/
# leakage_rules) or BP4's (journey_definition/scope_boundaries) - see the markdown cell's
# "Why this notebook is a genuinely different animal" section for the reasoning.
# ============================================================
CITATION_EVIDENCE_SCHEMA_FIELDS = {
    "evidence_id": "A stable identifier for one cited piece of evidence within a single BP6 "
    "recommendation - unique within that recommendation, never reused across recommendations.",
    "source_bp": "Which upstream BP (bp1-bp5) this evidence was retrieved from.",
    "source_gate": "Which gate of that upstream BP produced the artifact this evidence was read "
    "from (e.g. 1, 3, 6) - never assumed to be Gate 1 by default.",
    "source_artifact_relative_path": "The real, project-root-relative file path this evidence was "
    "read from (e.g. notebooks/bp1_customer_intent_classification/artifacts/policy.json) - always "
    "a real path to a real file that exists at retrieval time, never a description.",
    "source_field_or_metric": "The exact field, column, or metric name inside that artifact this "
    "evidence value was read from - never paraphrased.",
    "extracted_value": "The real value read from that field at retrieval time, copied verbatim "
    "(numbers as numbers, strings as strings) - never summarized or rounded without saying so.",
    "retrieval_timestamp_utc": "When this evidence was actually retrieved, in ISO-8601 UTC - so a "
    "later audit can tell whether the upstream artifact has since changed.",
    "verification_method": "How this evidence's presence in the source artifact was confirmed "
    "(e.g. 'read directly from JSON key path X') - never 'assumed' or 'recalled'.",
}

policy = {
    "bp_id": "bp6",
    "bp_name": "bp6_genai_resolution_assistant",
    "gate": 1,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "scope_definition": {
        "purpose": "Retrieve structured evidence from BP1-BP5's real outputs, summarize it, and "
        "recommend a next action with a citation/evidence field on every claim, per Master Plan "
        "Section 5.1/7 - human-in-the-loop approval required before any recommendation is treated "
        "as final.",
        "genai_call_occurs_at_this_gate": False,
        "genai_call_first_occurs_at_gate": GENAI_CALL_FIRST_OCCURS_AT_GATE,
        "this_gate_establishes": "Scope, real upstream-dependency status (live-checked), and the "
        "GenAI governance policy every later gate's output must satisfy - no retrieval, no "
        "generation, no external API call of any kind happens in this notebook.",
    },
    "upstream_dependency_policy": upstream_dependency_policy,
    "genai_usage_policy": {
        "grounding_definition": "A BP6 claim is 'grounded' only if every factual statement in it "
        "carries at least one citation/evidence record (schema below) pointing to a real field in "
        "a real upstream BP artifact - never an ungrounded free-text generation presented as fact.",
        "citation_evidence_schema": CITATION_EVIDENCE_SCHEMA_FIELDS,
        "human_in_the_loop": {
            "required": True,
            "auto_apply_allowed": False,
            "description": "No BP6 recommendation is ever applied, sent, or treated as final "
            "without an explicit human approval step - Master Plan Section 5.1's own words: "
            "'human-in-the-loop approval gate before any recommendation is treated as final.'",
        },
        "udaap_language_review": {
            "required": True,
            "applies_to": "Every GenAI-drafted customer-facing recommendation, before human "
            "approval (Master Plan Section 9, UDAAP row).",
            "description": "Reviewed for deceptive/misleading language before a human may approve "
            "it - this review is separate from, and precedes, the human-in-the-loop approval step "
            "above.",
        },
        "nist_ai_rmf": {
            "functions_mapped": "Govern / Map / Measure / Manage mapped onto Gates 1-6 (Master "
            "Plan Section 9, NIST AI RMF row) - this Gate 1 notebook is the 'Govern' function for "
            "BP6: the policy itself, not yet a Measure/Manage check on a real GenAI output.",
            "risk_category_field_required": True,
            "risk_category_value": "TBD_PENDING_FIRST_REAL_GATE5_GENAI_OUTPUT",
            "description": "Every GenAI output requires human-in-loop approval plus a documented "
            "risk category - the field is declared here (Govern); it is populated with a real "
            "value only once a real Gate 5 output exists to categorize (Measure/Manage).",
        },
        "glba_pii_masking": {
            "required": True,
            "applies_before": "Any GenAI call (BP6) or external API use (Gate 2 compliance "
            "touchpoint, Master Plan Section 8's gate table) - stated as policy here; the actual "
            "screen is a Gate 2 compliance touchpoint, not performed by this Gate 1 notebook.",
            "in_scope_narrative_text_source": "BANKING77's real 'text' field, surfaced via BP1's "
            "evidence - the only real narrative-text source in this suite, since the real CFPB "
            "extract carries no narrative-text column (Section 4 above, RAW_DATA_MANIFEST.md "
            "Finding 2).",
            "description": "Any PII surfaced in narrative text evidence is screened and masked "
            "before that text is placed into a GenAI prompt or returned in a GenAI output.",
        },
    },
    "grounding_integrity_rules": [
        "No BP6 recommendation may be treated as final without at least one citation/evidence "
        "record per factual claim, each conforming to the schema above - verified structurally "
        "below (citation_evidence_schema_has_required_fields).",
        "No BP6 recommendation may be auto-applied - human_in_the_loop.required and "
        "human_in_the_loop.auto_apply_allowed are both verified structurally below.",
        "Evidence retrieved from a CFPB-derived upstream BP (e.g. BP3, BP4, BP5) and evidence "
        "retrieved from a BANKING77-derived upstream BP (e.g. BP1) must never be silently merged "
        "into one row via a fabricated join key - each citation record references its own source "
        "BP's real key independently (Section 4 above: zero shared columns between the two real "
        "schemas, so no identifier-based join is possible by construction).",
        "This notebook itself makes zero external API calls and loads zero GenAI SDK module - "
        "verified live in Section 6 (genai_sdk_modules_loaded_this_run must be empty).",
    ],
    "compliance_touchpoints": {
        "gate1_baseline": {
            "requirement": "Data-minimization & purpose-limitation statement (GLBA/GDPR-aligned) "
            "- Master Plan Section 8's own Gate 1 row, applied to every BP.",
            "statement": "This Gate 1 notebook processes only real column NAMES (CFPB's 15-column "
            "schema, read as metadata, never row-level CFPB data) and the real BANKING77 `text`/"
            "`category` fields (anonymized customer-service utterances, no real-customer PII), for "
            "the stated purpose of recording BP6's scope and GenAI governance policy. No BP1-BP5 "
            "row-level output is retrieved, read, or processed by this notebook.",
        },
        "udaap": "See genai_usage_policy.udaap_language_review above (Master Plan Section 9).",
        "nist_ai_rmf": "See genai_usage_policy.nist_ai_rmf above (Master Plan Section 9).",
        "glba": "See genai_usage_policy.glba_pii_masking above (Master Plan Section 9).",
    },
    "assumptions": [
        "BP6 retrieves evidence from BP1-BP5's real outputs (Master Plan Section 5.1) - this "
        "notebook does not require those outputs to exist yet to define this policy; it records, "
        "live, which of the five already have a real Gate 1 policy.json today (Section 7 above).",
        "The real CFPB extract used in this project carries no narrative-text column "
        "(RAW_DATA_MANIFEST.md Finding 2, re-verified live in Section 4) - BANKING77's real "
        "`text` field, surfaced only via BP1's own evidence, is the sole real narrative-text "
        "source the GLBA PII-masking policy above needs to cover.",
        "BP6's citation/evidence schema is deliberately generic (source_bp/source_gate/"
        "source_artifact_relative_path/source_field_or_metric) rather than hardcoded to any "
        "upstream BP's internal field names, so it remains valid even as BP2/BP5's own schemas "
        "continue to evolve before BP6's own Gate 5 first retrieves from them.",
        "The NIST AI RMF risk-category value cannot be populated with a real category until a "
        "real Gate 5 GenAI output exists to categorize - recorded here as a required field with "
        "an explicit TBD value, never a guessed category.",
        "src/utils/bp1_config_sync.py is reused unmodified for BP6's own config file - already "
        "fully generic, parameterized by config_path, already reused unmodified by BP2/BP3/BP4.",
    ],
    "live_checks": {
        "cfpb_columns": sorted(cfpb_columns),
        "cfpb_narrative_text_columns": narrative_text_columns,
        "shared_columns_cfpb_banking77": sorted(shared_columns),
        "banking77_train_rows": train.height,
        "banking77_test_rows": test.height,
        "banking77_train_test_exact_text_overlap_rows": len(overlap_texts),
        "banking77_class_balance": banking77_class_balance,
        "genai_sdk_modules_loaded_this_run": genai_sdk_modules_loaded,
    },
}

# ============================================================
# SECTION 9: Write outputs (idempotent overwrite-in-place)
# ============================================================
policy_json_path = ARTIFACTS_DIR / "policy.json"
with open(policy_json_path, "w", encoding="utf-8") as f:
    json.dump(policy, f, indent=2)
print(f"\n[SAVED] {policy_json_path.relative_to(PROJECT_ROOT)}")

bp6_config_path = CONFIGS_DIR / "bp6_genai_resolution_assistant.yaml"

# Gate 1 owns ONLY the front-matter section of this shared config file - Gates 2-6 each own exactly
# one marker-delimited block below it, exactly as BP1/BP4's own config files work (see
# src/utils/bp1_config_sync.py's own module docstring for the real incident this pattern fixes,
# LESSONS_LEARNED_APPLIED.md #20). write_front_matter() replaces only this section and preserves
# every existing gate block verbatim regardless of position or order.
from utils.bp1_config_sync import read_existing_gate_block_markers, write_front_matter  # noqa: E402

_existing_gate_markers = read_existing_gate_block_markers(bp6_config_path)
_status_suffix = ""
for _gate_num, _gate_label in ((2, "Gate 2"), (3, "Gate 3"), (4, "Gate 4"), (5, "Gate 5"), (6, "Gate 6")):
    if any(_gate_label in _m for _m in _existing_gate_markers):
        _status_suffix += f"_gate{_gate_num}_confirmed"

_upstream_status_yaml_lines = "\n".join(
    f'  {bp_id}_{row["bp_name"][4:]}: "{row["config_status_string_live"] or "UNKNOWN"}"'
    for bp_id, row in upstream_dependency_policy.items()
)

bp6_config_text = f"""# Per-BP config - filled in at Gate 1 (Business Understanding & Policy)
# Gate 1 owns bp_id through random_state below via write_front_matter() (src/utils/bp1_config_sync.py,
# reused as-is from BP1/BP2/BP3/BP4 - fully generic, parameterized by config_path); Gates 2-6 each own
# exactly one marker-delimited block appended after it via write_gate_block() - do not hand-edit either
# section, re-run the owning notebook instead.
# BP6 deliberately does NOT reuse BP1-4's target_definition/leakage_rules schema (see this gate's own
# notebook markdown cell): BP6 has no supervised target and no training split at any gate - it is a
# retrieval-and-grounded-generation governance layer. genai_governance_policy and
# upstream_dependency_status replace those two keys.
bp_id: "bp6"
bp_name: "bp6_genai_resolution_assistant"
status: "gate1_confirmed{_status_suffix}"   # not_started|gate1|gate2|gate3|gate4|gate5|gate6_complete
target_definition: null   # BP6 has no supervised target at any gate - see genai_governance_policy below
genai_governance_policy:
  genai_call_occurs_at_gate: {GENAI_CALL_FIRST_OCCURS_AT_GATE}
  human_in_the_loop_required: true
  human_in_the_loop_auto_apply_allowed: false
  udaap_language_review_required: true
  nist_ai_rmf_risk_category_field_required: true
  nist_ai_rmf_risk_category_value: "TBD_PENDING_FIRST_REAL_GATE5_GENAI_OUTPUT"
  glba_pii_masking_required_before_genai_call: true
  citation_evidence_schema_fields:
    - evidence_id
    - source_bp
    - source_gate
    - source_artifact_relative_path
    - source_field_or_metric
    - extracted_value
    - retrieval_timestamp_utc
    - verification_method
upstream_dependency_status:
{_upstream_status_yaml_lines}
assumptions:
  - "BP6 retrieves evidence from BP1-BP5's real outputs (Master Plan Section 5.1); this notebook
     records, live, which of the five already have a real Gate 1 policy.json today rather than
     requiring all five to exist before BP6's own scope can be defined."
  - "The real CFPB extract carries no narrative-text column (RAW_DATA_MANIFEST.md Finding 2,
     re-verified live) - BANKING77's real 'text' field, surfaced only via BP1's own evidence, is
     the sole real narrative-text source the GLBA PII-masking policy needs to cover."
  - "BP6's citation/evidence schema is generic (source_bp/source_gate/source_artifact_relative_path/
     source_field_or_metric), never hardcoded to any upstream BP's internal field names, so it stays
     valid as BP2/BP5's own schemas continue to evolve before BP6's own Gate 5 first retrieves from
     them."
  - "The NIST AI RMF risk-category value cannot be populated with a real category until a real
     Gate 5 GenAI output exists to categorize - recorded as a required field with an explicit TBD
     value, never a guessed category."
random_state: 42
"""
write_front_matter(bp6_config_path, bp6_config_text)
print(f"[SAVED] {bp6_config_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 10: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
_citation_schema = policy["genai_usage_policy"]["citation_evidence_schema"]
_human_in_loop = policy["genai_usage_policy"]["human_in_the_loop"]
_udaap = policy["genai_usage_policy"]["udaap_language_review"]
_nist = policy["genai_usage_policy"]["nist_ai_rmf"]
_glba = policy["genai_usage_policy"]["glba_pii_masking"]
_required_citation_fields = {
    "evidence_id",
    "source_bp",
    "source_gate",
    "source_artifact_relative_path",
    "source_field_or_metric",
    "extracted_value",
    "retrieval_timestamp_utc",
    "verification_method",
}

checks = {
    "no_shared_identifier_column_cfpb_banking77": len(shared_columns) == 0,
    "cfpb_no_narrative_text_column": len(narrative_text_columns) == 0,
    "banking77_train_test_zero_exact_text_overlap": len(overlap_texts) == 0,
    "banking77_all_77_categories_present": len(b77_categories) == 77,
    "zero_genai_sdk_modules_loaded_this_run": len(genai_sdk_modules_loaded) == 0,
    "upstream_dependency_policy_covers_all_five_upstream_bps": set(upstream_dependency_policy.keys())
    == {"bp1", "bp2", "bp3", "bp4", "bp5"},
    "upstream_dependency_readiness_is_live_not_hardcoded": all(
        isinstance(row["gate1_policy_artifact_exists_live"], bool)
        for row in upstream_dependency_policy.values()
    ),
    "citation_evidence_schema_has_all_required_fields": (
        _required_citation_fields <= set(_citation_schema.keys())
    ),
    "human_in_the_loop_required_is_true": _human_in_loop["required"] is True,
    "human_in_the_loop_auto_apply_is_false": _human_in_loop["auto_apply_allowed"] is False,
    "udaap_review_required_is_true": _udaap["required"] is True,
    "nist_ai_rmf_risk_category_field_present": (
        "risk_category_value" in _nist and "risk_category_field_required" in _nist
    ),
    "glba_pii_masking_required_is_true": _glba["required"] is True,
    "genai_call_not_performed_at_this_gate": (
        policy["scope_definition"]["genai_call_occurs_at_this_gate"] is False
    ),
    "policy_json_written": policy_json_path.exists(),
    "bp6_config_yaml_written": bp6_config_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    "\n[ALL CHECKS PASSED] BP6 Gate 1 complete - scope defined, real upstream-dependency status "
    "live-checked (BP1-BP4 artifacts confirmed present/absent live, never hardcoded), GenAI "
    "governance policy recorded (citation schema, human-in-the-loop, UDAAP, NIST AI RMF, GLBA), "
    "zero external API calls made. Proceed to BP6 Gate 2 next, once BP6's own upstream retrieval "
    "design is ready to build."
)
